![Banner](img/AI_Special_Program_Banner.jpg)

## Recurrent Neural Networks (RNN) - Exercise 1: Tiny Language Model for Limericks
---
Instructions are given in $\color{teal}{\textsf{teal}}$ color (to be readable both on a dark as well as on a bright background).

In this exercise, we want to deal with a common requirement when working with deep learning models: Models should be reusable.

As you may have noticed, depending on the complexity of the problem and the amount of data, the models' training can take quite some time (when done on a CPU). Therefore, the ability to save and load already trained models is crucial to develop efficient problem solutions.

Additionally, we want to make use of the possibility to run notebooks in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb). The inherent advantage - given over a local execution - is that Colab allows the use of GPUs and, consequently, training time should significantly speed up.

<font color="teal"><b>Note:</b></font> For this exercise, you will need a Google account. If you don't already have one, please <font color="teal"><b>create one</b></font> using [this website](https://accounts.google.com/signup/v2/webcreateaccount?hl=en&flowName=GlifWebSignIn&flowEntry=SignUp). If you are not comfortable in providing your personal information, don't do so. You can delete your account and associated data after completing the exercise by following [these](https://support.google.com/accounts/answer/32046?hl=en) instructions.

As for the actual contents of this exercise, we will once more look at character-level language modeling with RNNs. In particular, we are trying to train a network capable of producing text that (hopefully) resembles the structure of so-called [Limericks](https://en.wikipedia.org/wiki/Limerick_(poetry)).

> Deep Thought, after millions of years,
<br>
> With the ultimate answer appears,
<br>
> &nbsp;&nbsp;&nbsp;&nbsp;Which is just 42.
<br>
> &nbsp;&nbsp;&nbsp;&nbsp;This is certainly true,
<br>
> Though it sounds a bit strange to the ears.

<div style="text-align: right"><a href="http://www.oedilf.com/db/Lim.php?Word=42">Source</a></div>

## $\color{teal}{\textsf{1. Setting up on Google Colab}}$

---

* $\color{teal}{\textsf{Upload this notebook to Google Colab}}$

    1. Visit https://colab.research.google.com/notebooks/intro.ipynb.
    2. Log in to your Google account.
    3. Select **`File`** **`Upload notebook`** from the menu bar.
    4. Upload this notebook.

* $\color{teal}{\textsf{Make sure to utilize GPUs}}$

    1. Select **`Edit`** **`Notebook settings`** from the menu bar.
    2. Choose *GPU* for hardware acceleration.

* $\color{teal}{\textsf{Set up directories for data and models}}$

    1. Open the file browser using the file icon in the navigation bar on the left hand side
    2. Right-click in the area below the folder called `sample_data` and choose *new folder*
    3. Create the folders `data` and `models`

* $\color{teal}{\textsf{Upload limericks}}$

  1. Using the menu to the right of the data folder (the three stacked dots appear as soon as you point your mouse to the data folder), upload the file `limericks.txt` from your local data folder

After these preparatory steps, you should be all set to train the model on Google Colab, once you have added the necessary code in this notebook (see below).

In [25]:
# Importing libraries
import numpy as np
import random
import os

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch import nn
from tqdm.notebook import trange, tqdm

import datetime as dt

In [26]:
import warnings
warnings.filterwarnings('ignore')

In [27]:
# limericks.zip -> data/limericks.txt
import zipfile, os

with zipfile.ZipFile('limericks.zip') as zf:
    zf.extractall('.')          # the archive already contains the data/ folder

print(os.listdir('data'))
print(f"data/limericks.txt: {os.path.getsize('data/limericks.txt') / 1024**2:.1f} MB")

['limericks.txt']
data/limericks.txt: 14.8 MB


## Preparations

This time, since we will be using colab, we will not import the helper functions we need, but instead we will have them here directly and use them as we did before. It will be very interesting to see which hardware acceleration is and which library versions are used on colab in particular.

In [28]:
def set_seed(seed=42):
    """
    Sets the random seed for reproducibility across Python, NumPy, and PyTorch.

    Parameters:
    seed (int): The seed value to use.
    """
    # 1. Set seed for Python's built-in random module
    random.seed(seed)

    # 2. Set seed for OS environment (affects Python hashing, e.g. in dictionaries)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # 3. Set seed for NumPy (this also ensures reproducibility in Pandas)
    np.random.seed(seed)

    # 4. Set seed for PyTorch (CPU and all GPUs)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For Multi-GPU setups

    # 5. Configure PyTorch/cuDNN for deterministic behavior
    # Note: This might reduce performance slightly but ensures consistent results
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"🌱 Random seed set to {seed}. Reproducibility mode enabled.")

RSEED = 42

# Example usage:
# Call this function right at the start of your notebook/script
set_seed(RSEED)

🌱 Random seed set to 42. Reproducibility mode enabled.


In [29]:
import sys
import types
import datetime
import platform

def print_execution_details(show_libs=True):
    """
    Prints a comprehensive report containing:
    1. Execution timestamp, OS, and Python version.
    2. Versions of all currently imported top-level libraries (optional).

    Parameters:
    show_libs (bool): If True, displays the versions of loaded libraries.
                      Default is True.
    """
    # --- PART 1: SYSTEM & TIME ---
    # Get current date and time
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Get Operating System details
    os_details = platform.platform() # Default fallback

    if platform.system() == "Linux":
        try:
            linux_info = platform.freedesktop_os_release()
            os_name = linux_info['NAME']
            os_version = linux_info['VERSION']
            os_details = f"{os_name} {os_version}".strip()
        except Exception:
            pass

    # Get Python version
    py_version = sys.version.split()[0]

    print(f"{'='*40}")
    print(f"EXECUTION DETAILS")
    print(f"{'='*40}\n")

    print(f"🕒 Run Start Time: {timestamp}")
    print(f"💻 Operating Sys:  {os_details}")
    print(f"🐍 Python Version: {py_version}")

    # --- PART 2: LIBRARY VERSIONS ---
    if show_libs:
        print(f"\n{'Loaded Library':<20} {'Version'}")
        print(f"{'-'*20} {'-'*15}")

        # Iterate over a copy of sys.modules to prevent runtime errors
        for name, module in sorted(list(sys.modules.items())):
            # Check if the item is a valid module
            if isinstance(module, types.ModuleType):
                # Safely attempt to access the __version__ attribute
                version = getattr(module, '__version__', None)

                # Filter: Show only top-level packages (no dots in name) that have a version
                if version and "." not in name:
                    print(f"{name:<20} {version}")
    else:
        print("\nℹ️  Library versions hidden (show_libs=False).")

    print(f"\n{'='*40}")

# Example 1: Show everything (Default)
print_execution_details(show_libs=True)

# Example 2: Compact mode
# print_execution_details(show_libs=False)

EXECUTION DETAILS

🕒 Run Start Time: 2026-07-27 07:42:04
💻 Operating Sys:  Ubuntu 22.04.5 LTS (Jammy Jellyfish)
🐍 Python Version: 3.12.13

Loaded Library       Version
-------------------- ---------------
IPython              7.34.0
PIL                  11.3.0
_csv                 1.0
_ctypes              1.1.0
_curses              b'2.2'
_decimal             1.70
anywidget            0.9.21
argparse             1.1
backcall             0.2.0
certifi              2026.06.17
csv                  1.0
ctypes               1.1.0
cuda                 12.9.7
cycler               0.12.1
dateutil             2.9.0.post0
debugpy              1.8.15
decimal              1.70
decorator            4.4.2
defusedxml           0.7.1
dill                 0.3.8
entrypoints          0.4
httplib2             0.32.0
ipaddress            1.0
ipykernel            6.17.1
ipython_genutils     0.2.0
ipywidgets           7.7.1
json                 2.0.9
jupyter_client       7.4.9
jupyter_core         5.9.1
kiwi

In [30]:
# --- CONFIGURATION ---
# Set this to True to disable GPU acceleration
USE_CPU = False

# Setup device-agnostic code
def get_default_device(force_cpu=False):
    """
    Selects the best available hardware.
    If force_cpu=True, CPU is always selected.
    """

    # 1. If CPU is forced, return immediately
    if force_cpu:
        return torch.device('cpu')

    # 2. Check for CUDA (NVIDIA) or ROCm (AMD)
    if torch.cuda.is_available():
        return torch.device('cuda')

    # 3. Check for MPS (Apple Silicon M1/M2/M3)
    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device('mps')

    # 4. Standard fallback to CPU (if no GPU is found)
    else:
        return torch.device('cpu')

def report_device(USE_CPU, device):
    # --- Status Outputs ---
    print(f"Active PyTorch Device: {device}\n" + "-"*30)

    if USE_CPU:
        print("ℹ️  NOTE: CPU mode was manually forced (USE_CPU=True) -> 🐢.")
        print("    Available GPUs are ignored.")

    elif device.type == 'cuda':
        print(f"🚀 Using GPU acceleration (CUDA/ROCm).")
        print(f"   Detected card: {torch.cuda.get_device_name(0)}")
        if torch.cuda.device_count() > 1:
            print(f"   (Note: {torch.cuda.device_count()} GPUs are available.)")

    elif device.type == 'mps':
        print("🍎 Using Apple Silicon acceleration (Metal Performance Shaders).")

    else:
        print("🐢 No hardware accelerator found.")
        print("    Training is running on CPU (standard fallback).")

# Set the device considering the flag
device = get_default_device(USE_CPU)

report_device(USE_CPU, device)

Active PyTorch Device: cuda
------------------------------
🚀 Using GPU acceleration (CUDA/ROCm).
   Detected card: Tesla T4


## 1. The Data: Limericks

The data we will work with is provided as simple continuous text. The limericks contained therein have been scraped from [this website](http://www.oedilf.com/db/Lim.php). You will find the text file `limericks.txt` in the `Data` folder next to this notebook.

In [31]:
# Reading data
limericks_file = "data/limericks.txt"
limericks = open(limericks_file, 'r').read()

**Remark:** If the above code cell throws an error, you need to go back to the setup above and make sure the limericks are uploaded to the `data` folder!

In [32]:
# Looking at some limericks
print(limericks[9990:10790])


from the mornings 'til mid-afternoons
in luzon you can feel the monsoons.
if that weren't enough,
here comes nastier stuff:
baguios, which are monstrous typhoons.

the old hay farmer's crib had backed up,
and resembled an overstuffed cup.
when he eyeballed the scene,
he said, "what does this mean?
there's a balefulness present, a-yup!"

with affamishment, food's what you need.
you would fight for a sesame seed.
a small crumb or an ort,
you'd take any old sort,
and you'd polish it off with great speed.

for bahaism, oneness is prime:
that of god, and of prophets in time,
of all peoples and races;
its primary place is
to end all our warfare and crime.

toss out all of those pepcid ac.
i am now achlorhydric. ah, me!
no more zantac or tums
in immeasurable sums ?
i'm a hulk of antacid debris.



## 2. Data Preparation
---

In [33]:
# Getting the unique characters used in the text
vocab = sorted(set(limericks))
print(vocab)
len(vocab)

['\t', '\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '^', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~']


70

In [34]:
# Creating unique index for each character in the text
char_to_index = {char:index for index, char in enumerate(vocab)}
list(char_to_index.items())[37:47]

[(']', 37),
 ('^', 38),
 ('_', 39),
 ('a', 40),
 ('b', 41),
 ('c', 42),
 ('d', 43),
 ('e', 44),
 ('f', 45),
 ('g', 46)]

In [35]:
# Checking index of a specific character
char_to_index['t']

59

In [36]:
# Mapping the index back to the characters
index_to_char = np.array(vocab)
print(index_to_char)

['\t' '\n' ' ' '!' '"' '#' '$' '%' '&' "'" '(' ')' '*' '+' ',' '-' '.' '/'
 '0' '1' '2' '3' '4' '5' '6' '7' '8' '9' ':' ';' '<' '=' '>' '?' '@' '['
 '\\' ']' '^' '_' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h' 'i' 'j' 'k' 'l' 'm' 'n'
 'o' 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z' '{' '|' '}' '~']


In [37]:
# Checking character for a given index
index_to_char[59]

np.str_('t')

In [38]:
# Encoding the entire text
encoded_text = np.array([char_to_index[char] for char in limericks])
encoded_text

array([42, 40, 55, ...,  3,  4,  1])

* $\color{teal}{\textsf{How many characters are there in the entire text?}}$

In [39]:
# Your solution goes here:
n_chars = len(limericks)

print(f"Characters in the entire text : {n_chars:,}")
print(f"Length of the encoded array   : {len(encoded_text):,}   (one integer per character)")
print(f"Size of the vocabulary        : {len(vocab)} distinct characters")

Characters in the entire text : 14,933,947
Length of the encoded array   : 14,933,947   (one integer per character)
Size of the vocabulary        : 70 distinct characters


In [40]:
# Comparing raw text ...
limericks[:100]

"cap'n jack was washed over the side.\nhis crew searched but found not hair nor hide.\nno longer the he"

In [41]:
# ... with encoded text
encoded_text[:100]

array([42, 40, 55,  9, 53,  2, 49, 40, 42, 50,  2, 62, 40, 58,  2, 62, 40,
       58, 47, 44, 43,  2, 54, 61, 44, 57,  2, 59, 47, 44,  2, 58, 48, 43,
       44, 16,  1, 47, 48, 58,  2, 42, 57, 44, 62,  2, 58, 44, 40, 57, 42,
       47, 44, 43,  2, 41, 60, 59,  2, 45, 54, 60, 53, 43,  2, 53, 54, 59,
        2, 47, 40, 48, 57,  2, 53, 54, 57,  2, 47, 48, 43, 44, 16,  1, 53,
       54,  2, 51, 54, 53, 46, 44, 57,  2, 59, 47, 44,  2, 47, 44])

The next thing we have to take care of is to split the data into batches. For this, it is important to determine a sequence length for each batch (number of characters) that

* is long enough to capture the general structure of a limerick, but
* is not too long, so that past information does not simply become noise.

* $\color{teal}{\textsf{Take a look at the limericks in \texttt{limericks.txt} and decide on a suitable \bf{sequence length}.}}$

In [42]:
# a limerick is a block of 5 lines, blocks are separated by a blank line -> how long is such a block?
limerick_lengths = [len(lim) for lim in limericks.split('\n\n') if lim.strip()]

print(f"{len(limerick_lengths):,} limericks in the file, median length {np.median(limerick_lengths):.0f} characters "
      f"(10%-90%: {np.percentile(limerick_lengths, 10):.0f}-{np.percentile(limerick_lengths, 90):.0f})")

sequence_length = 160  # <- your solution goes here

print(f"\nsequence_length = {sequence_length} covers roughly one complete limerick, so the network can pick up the "
      f"5-line layout\nand the AABBA rhyme scheme, while the context stays short enough not to drag in several "
      f"unrelated poems.")

90,528 limericks in the file, median length 163 characters (10%-90%: 148-178)

sequence_length = 160 covers roughly one complete limerick, so the network can pick up the 5-line layout
and the AABBA rhyme scheme, while the context stays short enough not to drag in several unrelated poems.


This is now used to create our data set of sequences of characters.

In [43]:
class CharSequenceDataset(Dataset):

    def __init__(self, text_encoded, seq_length, offset):
        sequences = []
        target = []
        for i in range(0, len(text_encoded) - seq_length - 1, offset):
            seq = text_encoded[i: i + seq_length]
            sequences.append(seq)
            target.append(text_encoded[i + seq_length])
        self.sequences = torch.tensor(sequences)
        self.target = torch.tensor(target)


    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.target[idx]


offset = sequence_length

ds = CharSequenceDataset(encoded_text, sequence_length, offset)

## inspection:
for seq, target in ds:
    print(seq.numpy(), ' -> ', target.numpy())
    print(repr(''.join(index_to_char[seq])),
          ' -> ', repr(''.join(index_to_char[target])))
    break

[42 40 55  9 53  2 49 40 42 50  2 62 40 58  2 62 40 58 47 44 43  2 54 61
 44 57  2 59 47 44  2 58 48 43 44 16  1 47 48 58  2 42 57 44 62  2 58 44
 40 57 42 47 44 43  2 41 60 59  2 45 54 60 53 43  2 53 54 59  2 47 40 48
 57  2 53 54 57  2 47 48 43 44 16  1 53 54  2 51 54 53 46 44 57  2 59 47
 44  2 47 44 51 52 14  1 41 60 59  2 59 47 44  2 43 44 44 55  2 41 44 53
 59 47 48 42  2 57 44 40 51 52 14  1 48 58  2 62 47 44 57 44  2 49 40 42
 50  2 62 48 51 51  2 45 54 57 44 61 44 57  2 57]  ->  44
"cap'n jack was washed over the side.\nhis crew searched but found not hair nor hide.\nno longer the helm,\nbut the deep benthic realm,\nis where jack will forever r"  ->  'e'


In [44]:
# final preparation step: create a data loader
BATCH_SIZE = 128

trainloader = DataLoader(ds, shuffle=True, batch_size=BATCH_SIZE)
len(ds)

93337

## 3. RNN Model Creation
---

In [45]:
# Vocabulary length
vocab_size = len(vocab)
# Embedding dimensions
embed_dim = 64
# Number of RNN units (neurons)
rnn_units = 512

In [46]:
class Network(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units, recurrent_type='LSTM'):
        super(Network, self).__init__()
        self.recurrent_type = recurrent_type
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        if recurrent_type == 'RNN':
            self.rnn = nn.RNN(embedding_dim, rnn_units, batch_first=True)
        elif recurrent_type == 'LSTM':
            self.rnn = nn.LSTM(embedding_dim, rnn_units, batch_first=True)
        elif recurrent_type == 'GRU':
            self.rnn = nn.GRU(embedding_dim, rnn_units, batch_first=True)
        self.linear = nn.Linear(rnn_units, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        #lstm also returns the cell hidden state
        if self.recurrent_type == "LSTM":
            output, (hidden, cell_hidden) = self.rnn(embedded)
        else:
            output, hidden = self.rnn(embedded)
        output = self.linear(hidden[-1])
        return output

# Instantiate the PyTorch model
model = Network(vocab_size, embed_dim, rnn_units, recurrent_type='LSTM')
model

Network(
  (embedding): Embedding(70, 64)
  (rnn): LSTM(64, 512, batch_first=True)
  (linear): Linear(in_features=512, out_features=70, bias=True)
)

In [47]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters())

Setup the training loop

In [48]:
def train(model, dataloader, optimizer, criterion):
    model.train()

    running_loss = 0.0

    # Iterate over the training dataset
    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device).long()
        # Zero the gradients
        optimizer.zero_grad()
        # Forward pass
        outputs = model(inputs)
        # Compute the loss
        loss = criterion(outputs, labels)
        # Backward pass
        loss.backward()
        # Update the weights
        optimizer.step()

        running_loss += loss.item()

    # Calculate the average loss for the epoch
    epoch_loss = running_loss / len(dataloader)
    return epoch_loss

def train_for_epochs(epchs, model, trainloader, optimizer, criterion):
    history = {
        "loss": []
    }
    loop = trange(epchs)
    # Training loop
    for epoch in loop:

        epoch_loss_train = train(model, trainloader, optimizer, criterion)

        history["loss"].append(epoch_loss_train)

        loop.set_description(f"Train Loss: {epoch_loss_train:.4f}")
    return history


* <font color="teal"> Train the model for <b>2</b> epochs. Double-check that you enabled a GPU runtime (unless you don't mind waiting). However, even if you are using GPUs, it may take a while to complete the training.</font>

In [49]:
# Your solution goes here:
model.to(device)   # the training loop pushes every batch to `device`, so the weights have to live there too

EPOCHS_1ST = 2

start_train = dt.datetime.now()
print(f"Training started : {start_train}")
history = train_for_epochs(EPOCHS_1ST, model, trainloader, optimizer, criterion)
finish_train = dt.datetime.now()
print(f"Training finished: {finish_train}  (duration: {finish_train - start_train})")

# collect the losses of all training runs in one list, so training can be resumed later on
hist_cum = list(history["loss"])
total_epochs = EPOCHS_1ST

print(f"\nTrained for {total_epochs} epochs - training loss went from {hist_cum[0]:.4f} to {hist_cum[-1]:.4f}.")

Training started : 2026-07-27 07:42:15.375266


  0%|          | 0/2 [00:00<?, ?it/s]

Training finished: 2026-07-27 07:43:38.024791  (duration: 0:01:22.649525)

Trained for 2 epochs - training loss went from 2.3546 to 1.9984.


* $\color{teal}{\textsf{Save the state dict of your model to your session storage.}}$

**Hint**: Have a look at the [documentation](https://pytorch.org/tutorials/beginner/saving_loading_models.html) to save and load models.

In [50]:
# Your solution goes here:
os.makedirs("models", exist_ok=True)

PATH_2_EPOCHS = "models/network.pt"
# recommended way: store only the learned parameters, not the pickled model object itself
torch.save(model.state_dict(), PATH_2_EPOCHS)

print(f"State dict after {total_epochs} epochs saved to '{PATH_2_EPOCHS}' "
      f"({os.path.getsize(PATH_2_EPOCHS) / 1024**2:.1f} MB).")

State dict after 2 epochs saved to 'models/network.pt' (4.7 MB).


* <font color="teal"> Continue training for another <b>18</b> epochs. In total you should have trained the model for <b>20</b> epochs.</font>

In [51]:
# Your solution goes here:
EPOCHS_2ND = 18

start_train = dt.datetime.now()
print(f"Training started : {start_train}")
# neither model nor optimizer are re-instantiated -> this really continues the previous training
history = train_for_epochs(EPOCHS_2ND, model, trainloader, optimizer, criterion)
finish_train = dt.datetime.now()
print(f"Training finished: {finish_train}  (duration: {finish_train - start_train})")

hist_cum += history["loss"]
total_epochs += EPOCHS_2ND

print(f"\nTrained for {total_epochs} epochs in total - training loss went from "
      f"{hist_cum[0]:.4f} (epoch 1) to {hist_cum[-1]:.4f} (epoch {total_epochs}).")

Training started : 2026-07-27 07:43:38.054984


  0%|          | 0/18 [00:00<?, ?it/s]

Training finished: 2026-07-27 07:56:57.239847  (duration: 0:13:19.184863)

Trained for 20 epochs in total - training loss went from 2.3546 (epoch 1) to 0.1896 (epoch 20).


* <font color="teal"> Save the state of of your model to your session storage, but use a different file name this time.</font>

In [52]:
# Your solution goes here:
PATH_20_EPOCHS = "models/network_v2.pt"   # different file name -> the 2-epoch model stays available
torch.save(model.state_dict(), PATH_20_EPOCHS)

print(f"State dict after {total_epochs} epochs saved to '{PATH_20_EPOCHS}' "
      f"({os.path.getsize(PATH_20_EPOCHS) / 1024**2:.1f} MB).")

print("\nModels in the session storage:")
for f in sorted(os.listdir("models")):
    print(f"  models/{f}")
print("\n-> download both files (file browser -> three dots next to the file -> Download) before closing "
      "the session,\n   they are loaded again in exercise 2.")

State dict after 20 epochs saved to 'models/network_v2.pt' (4.7 MB).

Models in the session storage:
  models/network.pt
  models/network_v2.pt

-> download both files (file browser -> three dots next to the file -> Download) before closing the session,
   they are loaded again in exercise 2.


* <font color="teal"> Download both saved models from your session storage to your local hard drive.</font>

**CAUTION**: If you accidentally close the current session without downloading the models first, those will be purged and your training progress is lost.

* <font color="teal"> Continue with exercise 2. (Either locally or in Colab is fine.)</font>

In [53]:
from google.colab import files

files.download('models/network.pt')
files.download('models/network_v2.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>